# Atividade 03 - IA
## Distância Euclidiana, Distância de Cosseno e Busca Semântica

---
### Instalação e configuração da API via OpenRouter

In [ ]:
!pip install -q openai

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get('OPENROUTER_API_KEY')
)

### Funções auxiliares

In [ ]:
MAX_CHARS = 6000  # Limite de caracteres por trecho para não estourar os 8192 tokens da API

def get_embedding(texto):
    """
    Gera o embedding de um texto usando a API do OpenRouter.
    Se o texto for uma tupla (rótulo, frase), usa apenas a frase.
    Trunca textos muito longos.
    """
    if isinstance(texto, tuple):
        texto = texto[1]
    texto = texto[:MAX_CHARS]
    response = client.embeddings.create(
        model="openai/text-embedding-3-small",
        input=texto
    )
    return response.data[0].embedding


def get_embeddings_batch(textos):
    """
    Gera embeddings para uma lista de textos de uma vez.
    Trunca cada texto ao limite de caracteres.
    """
    textos_truncados = [t[:MAX_CHARS] for t in textos]
    response = client.embeddings.create(
        model="openai/text-embedding-3-small",
        input=textos_truncados
    )
    return [item.embedding for item in response.data]

---
## Parte 1 — Funções de Distância

### 1. Distância Euclidiana

$$d = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

In [ ]:
import math
import numpy as np

def distancia_euclidiana(embedding_a, embedding_b):
    """
    Calcula a distância euclidiana entre dois embeddings.
    Recebe dois vetores de qualquer dimensão, desde que possuam o mesmo tamanho.
    """
    if len(embedding_a) != len(embedding_b):
        raise ValueError("Os dois embeddings devem possuir a mesma dimensão.")

    soma = sum((a - b) ** 2 for a, b in zip(embedding_a, embedding_b))
    return math.sqrt(soma)

### 2. Similaridade de Cosseno e Distância de Cosseno

$$sim = \frac{A \cdot B}{\|A\| \times \|B\|}$$

$$d = 1 - sim$$

In [ ]:
def similaridade_cosseno(embedding_a, embedding_b):
    """
    Calcula a similaridade de cosseno entre dois embeddings.
    Retorna um valor entre -1 e 1 (1 = idênticos, 0 = ortogonais).
    """
    if len(embedding_a) != len(embedding_b):
        raise ValueError("Os dois embeddings devem possuir a mesma dimensão.")

    produto_escalar = sum(a * b for a, b in zip(embedding_a, embedding_b))
    norma_a = math.sqrt(sum(a ** 2 for a in embedding_a))
    norma_b = math.sqrt(sum(b ** 2 for b in embedding_b))

    if norma_a == 0 or norma_b == 0:
        raise ValueError("Não é possível calcular para vetores nulos.")

    return produto_escalar / (norma_a * norma_b)


def distancia_cosseno(embedding_a, embedding_b):
    """
    Calcula a distância de cosseno entre dois embeddings.
    distância = 1 - similaridade
    """
    return 1 - similaridade_cosseno(embedding_a, embedding_b)

### 3. Teste com vetores simples

In [ ]:
embedding_a = [1, 0, 0]
embedding_b = [0, 1, 0]
embedding_c = [1, 0, 0]

print("Embeddings de teste:")
print(f"  embedding_a = {embedding_a}")
print(f"  embedding_b = {embedding_b}")
print(f"  embedding_c = {embedding_c}")

print("\n>> embedding_a x embedding_b")
print(f"   Dist. Euclidiana: {distancia_euclidiana(embedding_a, embedding_b):.4f}")
print(f"   Dist. Cosseno:    {distancia_cosseno(embedding_a, embedding_b):.4f}")

print("\n>> embedding_a x embedding_c")
print(f"   Dist. Euclidiana: {distancia_euclidiana(embedding_a, embedding_c):.4f}")
print(f"   Dist. Cosseno:    {distancia_cosseno(embedding_a, embedding_c):.4f}")

print("\n>> embedding_b x embedding_c")
print(f"   Dist. Euclidiana: {distancia_euclidiana(embedding_b, embedding_c):.4f}")
print(f"   Dist. Cosseno:    {distancia_cosseno(embedding_b, embedding_c):.4f}")

### 4. Teste com embeddings reais e geração do gráfico 3D

In [ ]:
import itertools

palavras_animais = ["gato", "cachorro", "felino"]
palavras_veiculos = ["carro", "moto", "caminhão"]
palavras_frutas = ["banana", "maçã", "goiaba"]

todas_palavras = palavras_animais + palavras_veiculos + palavras_frutas

# Gera os embeddings via OpenRouter
response = client.embeddings.create(
    model="openai/text-embedding-3-small",
    input=todas_palavras
)

embs_palavras = {}
for palavra, item in zip(todas_palavras, response.data):
    embs_palavras[palavra] = item.embedding

# Tabela comparativa
pares = list(itertools.combinations(todas_palavras, 2))
print(f"{'Par':<25} {'Dist. Euclidiana':>18} {'Dist. Cosseno':>15}")
print("-" * 60)
for p1, p2 in pares:
    d_euc = distancia_euclidiana(embs_palavras[p1], embs_palavras[p2])
    d_cos = distancia_cosseno(embs_palavras[p1], embs_palavras[p2])
    print(f"{p1 + ' x ' + p2:<25} {d_euc:>18.4f} {d_cos:>15.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D

nomes = list(embs_palavras.keys())
matriz = np.array([embs_palavras[p] for p in nomes])

pca = PCA(n_components=3)
coords_3d = pca.fit_transform(matriz)

cores = []
for palavra in nomes:
    if palavra in palavras_animais:
        cores.append('#4285F4')
    elif palavra in palavras_veiculos:
        cores.append('#EA4335')
    else:
        cores.append('#34A853')

fig = plt.figure(figsize=(12, 9), facecolor='white')
ax = fig.add_subplot(111, projection='3d')
ax.scatter(coords_3d[:, 0], coords_3d[:, 1], coords_3d[:, 2], c=cores, s=120, depthshade=True)

for i, nome in enumerate(nomes):
    ax.text(coords_3d[i, 0], coords_3d[i, 1], coords_3d[i, 2], '  ' + nome, fontsize=11)

ax.set_title('Embeddings de palavras reduzidos para 3D (via PCA)', fontsize=14)
ax.set_xlabel('Componente 1')
ax.set_ylabel('Componente 2')
ax.set_zlabel('Componente 3')
plt.tight_layout()
plt.show()

---
## Parte 2 — Comparação de Frases (Frase Âncora)

In [ ]:
import pandas as pd

frase_ancora = "O cachorro correu no parque e brincou com a bola."

frases_comparacao = [
    ("Similar (mesmo sentido, palavras diferentes)", "Um cão estava correndo no jardim e brincando com seu brinquedo."),
    ("Relacionado (mesmo contexto de animais)", "O gato dormiu na almofada da sala durante toda a tarde."),
    ("Diferente (outro domínio - economia)", "A taxa de juros do banco central subiu dois pontos percentuais."),
    ("Oposto/Negação", "Nenhum animal esteve no parque e o cão permaneceu preso em casa.")
]

# 1. Gerar o embedding da âncora
vec_ancora = np.array(get_embedding(frase_ancora), dtype=np.float32)

# 2. Gerar os embeddings das comparações
vecs_comp = []
for rotulo, frase in frases_comparacao:
    vec = np.array(get_embedding(frase), dtype=np.float32)
    vecs_comp.append(vec)

# 3. Montar tabela de resultados
resultados = []
for (rotulo, frase), vec in zip(frases_comparacao, vecs_comp):
    resultados.append({
        "Categoria": rotulo,
        "Frase": frase,
        "Dist. Euclidiana": round(distancia_euclidiana(vec_ancora, vec), 4),
        "Similaridade Cosseno": round(similaridade_cosseno(vec_ancora, vec), 4),
        "Distância Cosseno": round(distancia_cosseno(vec_ancora, vec), 4)
    })

df_resultados = pd.DataFrame(resultados)
print(f'Frase âncora: "{frase_ancora}"\n')
df_resultados

---
## Parte 3 — Busca Semântica Simples

Lemos os arquivos markdown, separamos em trechos, geramos embeddings e encontramos os TOP 3 mais similares a uma query.

### Upload dos arquivos markdown

In [ ]:
from google.colab import files

print("Faça upload dos arquivos .md da Aula 2:")
uploaded = files.upload()

In [ ]:
# Lê todos os arquivos enviados
documentos = {}
for nome_arquivo, conteudo in uploaded.items():
    documentos[nome_arquivo] = conteudo.decode('utf-8')
    print(f"Arquivo carregado: {nome_arquivo} ({len(conteudo)} bytes)")

### Função de busca semântica

In [ ]:
import time

def busca_semantica(query, trechos, top_k=3):
    """
    Recebe uma query e uma lista de trechos de texto.
    Gera embeddings, calcula similaridade de cosseno e retorna o TOP K.
    Trunca trechos longos para não estourar o limite da API.
    """
    # Embedding da query
    vec_query = np.array(get_embedding(query), dtype=np.float32)

    resultados = []
    # Gera embeddings em lotes de 10 para não sobrecarregar a API
    batch_size = 10
    for i in range(0, len(trechos), batch_size):
        batch_original = trechos[i:i+batch_size]
        # Trunca cada trecho ao limite de caracteres
        batch_truncado = [t[:MAX_CHARS] for t in batch_original]
        response = client.embeddings.create(
            model="openai/text-embedding-3-small",
            input=batch_truncado
        )
        for j, item in enumerate(response.data):
            vec = np.array(item.embedding, dtype=np.float32)
            sim = similaridade_cosseno(vec_query, vec)
            resultados.append({
                "Trecho": batch_original[j],
                "Similaridade": round(sim, 4)
            })
        time.sleep(1)  # Evita rate limit

    # Ordena por similaridade (maior = mais similar)
    resultados.sort(key=lambda x: x["Similaridade"], reverse=True)
    return resultados[:top_k]

### Busca por LINHA

In [ ]:
# Separa todos os documentos linha por linha
todas_linhas = []
for nome, conteudo in documentos.items():
    linhas = [linha.strip() for linha in conteudo.split('\n') if linha.strip() and len(linha.strip()) > 10]
    todas_linhas.extend(linhas)

print(f"Total de linhas: {len(todas_linhas)}")

In [ ]:
query = 'O que é autonomia e opacidade algorítmica?'
print(f'Query: "{query}"\n')
print('TOP 3 linhas mais similares:')
print('-' * 80)

top_linhas = busca_semantica(query, todas_linhas, top_k=3)
for i, r in enumerate(top_linhas, 1):
    print(f"\n{i}. [Similaridade: {r['Similaridade']}]")
    print(f"   {r['Trecho'][:200]}")

In [ ]:
query2 = 'O que é o diário de bordo da IA?'
print(f'Query: "{query2}"\n')
print('TOP 3 linhas mais similares:')
print('-' * 80)

top_linhas2 = busca_semantica(query2, todas_linhas, top_k=3)
for i, r in enumerate(top_linhas2, 1):
    print(f"\n{i}. [Similaridade: {r['Similaridade']}]")
    print(f"   {r['Trecho'][:200]}")

### Busca por PARÁGRAFO

In [ ]:
# Separa todos os documentos por parágrafos (blocos separados por linhas em branco)
todos_paragrafos = []
for nome, conteudo in documentos.items():
    blocos = conteudo.split('\n\n')
    paragrafos = [bloco.strip() for bloco in blocos if bloco.strip() and len(bloco.strip()) > 20]
    todos_paragrafos.extend(paragrafos)

print(f"Total de parágrafos: {len(todos_paragrafos)}")

In [ ]:
query = 'O que é autonomia e opacidade algorítmica?'
print(f'Query: "{query}"\n')
print('TOP 3 parágrafos mais similares:')
print('-' * 80)

top_paragrafos = busca_semantica(query, todos_paragrafos, top_k=3)
for i, r in enumerate(top_paragrafos, 1):
    print(f"\n{i}. [Similaridade: {r['Similaridade']}]")
    print(f"   {r['Trecho'][:300]}")

In [ ]:
query2 = 'O que é o diário de bordo da IA?'
print(f'Query: "{query2}"\n')
print('TOP 3 parágrafos mais similares:')
print('-' * 80)

top_paragrafos2 = busca_semantica(query2, todos_paragrafos, top_k=3)
for i, r in enumerate(top_paragrafos2, 1):
    print(f"\n{i}. [Similaridade: {r['Similaridade']}]")
    print(f"   {r['Trecho'][:300]}")

### Busca por CAPÍTULO

In [ ]:
import re

# Separa todos os documentos por capítulos (cabeçalhos markdown: # ou ##)
todos_capitulos = []
for nome, conteudo in documentos.items():
    partes = re.split(r'(?=^#{1,2}\s)', conteudo, flags=re.MULTILINE)
    capitulos = [parte.strip() for parte in partes if parte.strip() and len(parte.strip()) > 30]
    todos_capitulos.extend(capitulos)

print(f"Total de capítulos: {len(todos_capitulos)}")
for i, cap in enumerate(todos_capitulos):
    titulo = cap.split('\n')[0]
    print(f"  {i+1}. {titulo[:80]}")

In [ ]:
query = 'O que é autonomia e opacidade algorítmica?'
print(f'Query: "{query}"\n')
print('TOP 3 capítulos mais similares:')
print('-' * 80)

top_capitulos = busca_semantica(query, todos_capitulos, top_k=3)
for i, r in enumerate(top_capitulos, 1):
    print(f"\n{i}. [Similaridade: {r['Similaridade']}]")
    print(f"   {r['Trecho'][:400]}")
    print(f"   ...")

In [ ]:
query2 = 'O que é o diário de bordo da IA?'
print(f'Query: "{query2}"\n')
print('TOP 3 capítulos mais similares:')
print('-' * 80)

top_capitulos2 = busca_semantica(query2, todos_capitulos, top_k=3)
for i, r in enumerate(top_capitulos2, 1):
    print(f"\n{i}. [Similaridade: {r['Similaridade']}]")
    print(f"   {r['Trecho'][:400]}")
    print(f"   ...")